In [1]:
import boto3
import io
import pandas as pd
import json

class S3DataFetcher:
    def __init__(self, aws_access_key_id=None, aws_secret_access_key=None, region_name="ap-southeast-1"):
        """
        Khởi tạo kết nối S3. Nếu bạn đã cấu hình qua AWS CLI, có thể bỏ qua key/secret.
        """
        self.s3 = boto3.client(
            "s3",
            region_name=region_name
        )

    def read_file(self, bucket_name: str, object_key: str, file_type: str = "csv"):
        """
        Tải nội dung file từ S3 và xử lý theo định dạng.
        
        :param bucket_name: tên bucket S3
        :param object_key: đường dẫn đầy đủ tới file (key)
        :param file_type: "csv", "json", hoặc "text"
        :return: DataFrame, dict/list, hoặc str tùy loại file
        """
        try:
            response = self.s3.get_object(Bucket=bucket_name, Key=object_key)
            content = response["Body"].read().decode("utf-8")
        except Exception as e:
            raise RuntimeError(f"Không thể tải file từ S3: {e}")

        if file_type == "csv":
            return pd.read_csv(io.StringIO(content))
        elif file_type == "json":
            return json.loads(content)
        elif file_type == "text":
            return content
        else:
            raise ValueError("file_type phải là 'csv', 'json', hoặc 'text'.")

    def list_files(self, bucket_name: str, prefix: str = ""):
        """
        Liệt kê tất cả file trong bucket hoặc một "thư mục con" (prefix)

        :param bucket_name: tên bucket
        :param prefix: đường dẫn thư mục con, ví dụ '2025/july/'
        :return: list các key (đường dẫn file)
        """
        try:
            response = self.s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
            if "Contents" not in response:
                return []
            return [obj["Key"] for obj in response["Contents"]]
        except Exception as e:
            raise RuntimeError(f"Lỗi khi liệt kê file: {e}")

if __name__ == "__main__":
    # Thông tin cấu hình
    bucket = "team253"
    key = "models/catboost_aml_model.pkl"  # ví dụ: "data/test.csv"
    file_type = "json"  # hoặc "json", "text"

    # Khởi tạo fetcher (dùng credentials đã cấu hình sẵn)
    fetcher = S3DataFetcher()

    # Đọc file
    try:
        result = fetcher.read_file(bucket_name=bucket, object_key=key, file_type=file_type)

        # In kết quả
        print("✅ File đọc thành công.")
        if file_type == "csv":
            print(result.head())
        elif file_type == "json":
            print(json.dumps(result, indent=2, ensure_ascii=False))
        else:
            print(result)
    except Exception as e:
        print(f"❌ Lỗi khi đọc file: {e}")

    # Liệt kê các file trong thư mục (tuỳ chọn test)
    try:
        files = fetcher.list_files(bucket_name=bucket, prefix="data/")
        print(f"📁 Có {len(files)} file:")
        for f in files:
            print("-", f)
    except Exception as e:
        print(f"❌ Lỗi khi liệt kê file: {e}")

❌ Lỗi khi đọc file: Không thể tải file từ S3: Unable to locate credentials
❌ Lỗi khi liệt kê file: Lỗi khi liệt kê file: Unable to locate credentials


In [ ]:
import boto3


def hello_s3():
    """
    Use the AWS SDK for Python (Boto3) to create an Amazon Simple Storage Service
    (Amazon S3) client and list the buckets in your account.
    This example uses the default settings specified in your shared credentials
    and config files.
    """
    import dotenv   
    import os
    dotenv.load_dotenv()
    aws_access_key_id = os.getenv("AWS_ACCESS_KEY")
    aws_secret_access_key = os.getenv("AWS_SECRET_KEY")
    region_name = os.getenv("AWS_REGION", "ap-southeast-1")
    # aws_access_key_id = aws_access_key_id or Utils.load_api_key_from_env("AWS_ACCESS_KEY")
    # aws_secret_access_key = aws_secret_access_key or Utils.load_api_key_from_env("AWS_SECRET_KEY")
    # Create an S3 client.
    s3_client = boto3.client(
                    's3',
                    region_name=region_name,
                    aws_access_key_id=aws_access_key_id,
                    aws_secret_access_key=aws_secret_access_key
                )

    response = s3_client.list_buckets()
    print("Buckets:")
    for bucket in response['Buckets']:
        print(f"  {bucket['Name']}")

if __name__ == "__main__":
    hello_s3()



Buckets:
  aws-cloudtrail-logs-048013208071-9bbb01c7
  sagemaker-ap-east-1-048013208071
  sagemaker-ap-southeast-1-048013208071
  sagemaker-ap-southeast-2-048013208071
  sagemaker-studio-048013208071-2hym88kbrgv
  sagemaker-studio-048013208071-665mwlxndwh
  sagemaker-studio-048013208071-ddsaoenlnuf
  sagemaker-studio-048013208071-eti8cnjez3i
  sagemaker-studio-048013208071-j41ntlm8zb
  sagemaker-studio-048013208071-zqz9n27xe2
  sagemaker-studio-c012qj1c0ej
  sagemaker-us-east-1-048013208071
  team253
